In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "5"
os.environ["HF_HOME"] = "/local1/mohsenfayyaz/.hfcache/"

!git -C ColBERT/ pull || git clone https://github.com/stanford-futuredata/ColBERT.git
import sys; sys.path.insert(0, 'ColBERT/')

try: # When on google Colab, let's install all dependencies with pip.
    import google.colab
    !pip install -U pip
    !pip install -e ColBERT/['faiss-gpu','torch']
except Exception:
  import sys; sys.path.insert(0, 'ColBERT/')
  try:
    from colbert import Indexer, Searcher
  except Exception:
    print("If you're running outside Colab, please make sure you install ColBERT in conda following the instructions in our README. You can also install (as above) with pip but it may install slower or less stable faiss or torch dependencies. Conda is recommended.")
    assert False

Already up to date.


# RUN ONCE

In [ ]:
import pandas as pd

# Login using e.g. `huggingface-cli login` to access this dataset
# brevity_bias
df_foil = pd.read_json("hf://datasets/mohsenfayyaz/ColDeR/test/brevity_bias.jsonl", lines=True)
df_foil.head(1)

In [ ]:
import torch
import numpy as np
from colbert.modeling.checkpoint import Checkpoint
from colbert.infra import ColBERTConfig
from colbert.modeling.colbert import colbert_score

checkpoint = 'colbert-ir/colbertv2.0'
config = ColBERTConfig(doc_maxlen=510, nbits=2)
ckpt = Checkpoint(checkpoint, colbert_config=config)

def get_colbert_scores(query, data):
    Q = ckpt.queryFromText([query])
    D = ckpt.docFromText(data, bsize=32)[0]
    D_mask = torch.ones(D.shape[:2], dtype=torch.long)
    scores = colbert_score(Q, D, D_mask).flatten().cpu().numpy().tolist()
    ranking = np.argsort(scores)[::-1]
    return scores, ranking

In [ ]:
from tqdm.auto import tqdm

rdf = []
for row in tqdm(df_foil.to_dict(orient="records")):
    query = row["query"]
    data = [row["document_1"], row["document_2"]]
    scores, ranking = get_colbert_scores(query, data)
    rdf.append({"query": query, "document_1": data[0], "document_2": data[1], "score_1": scores[0], "score_2": scores[1], "ranking": ranking})
    
rdf = pd.DataFrame(rdf)
rdf

In [ ]:
from scipy import stats

def standard_ttest_ppf(n, confidence_level=0.95):
    return stats.t.ppf(q=1-confidence_level, df=n-1, loc=0, scale=1)

df = rdf.copy()
col1, col2 = "score_2", "score_1"
ttest = stats.ttest_rel(df[col1], df[col2])
result = {
    "col1": col1,
    "col2": col2,
    "ttest_stats": ttest[0],
    "ttest_pvalue": ttest[1],
    "ttest_ci_low_stats": ttest.confidence_interval(confidence_level=0.95)[0],
    "ttest_ci_high_stats": ttest.confidence_interval(confidence_level=0.95)[1],
    "ttest_ci_low": np.abs(standard_ttest_ppf(len(df))),
    "ttest_ci_high": np.abs(standard_ttest_ppf(len(df))),
    "standard_ttest_ppf": standard_ttest_ppf(len(df)),
    "acc": (df[col1] > df[col2]).mean(),
}
result

# RUN ALL

In [2]:
import torch
import numpy as np
from colbert.modeling.checkpoint import Checkpoint
from colbert.infra import ColBERTConfig
from colbert.modeling.colbert import colbert_score

checkpoint = 'colbert-ir/colbertv2.0'
config = ColBERTConfig(doc_maxlen=510, nbits=2)
ckpt = Checkpoint(checkpoint, colbert_config=config)

def get_colbert_scores(query, data):
    Q = ckpt.queryFromText([query])
    D = ckpt.docFromText(data, bsize=32)[0]
    D_mask = torch.ones(D.shape[:2], dtype=torch.long)
    scores = colbert_score(Q, D, D_mask).flatten().cpu().numpy().tolist()
    ranking = np.argsort(scores)[::-1]
    return scores, ranking

/data2/mohsenfayyaz/projects/Retriever-Contextualization/src/notebooks/Rebuttal/ColBERT/colbert/utils/amp.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()


In [3]:
import pandas as pd
from tqdm.auto import tqdm
from scipy import stats


def standard_ttest_ppf(n, confidence_level=0.95):
    return stats.t.ppf(q=1-confidence_level, df=n-1, loc=0, scale=1)

results = []
for part in tqdm(["brevity_bias", "answer_importance", "repetition_bias", "position_bias", "literal_bias"]):
    df_foil = pd.read_json(f"hf://datasets/mohsenfayyaz/ColDeR/test/{part}.jsonl", lines=True)

    rdf = []
    for row in tqdm(df_foil.to_dict(orient="records")):
        query = row["query"]
        data = [row["document_1"], row["document_2"]]
        scores, ranking = get_colbert_scores(query, data)
        rdf.append({"query": query, "document_1": data[0], "document_2": data[1], "score_1": scores[0], "score_2": scores[1], "ranking": ranking})
        
    rdf = pd.DataFrame(rdf)
    
    df = rdf.copy()
    col1, col2 = "score_1", "score_2"
    ttest = stats.ttest_rel(df[col1], df[col2])
    result = {
        "col1": col1,
        "col2": col2,
        "ttest_stats": ttest[0],
        "ttest_pvalue": ttest[1],
        "ttest_ci_low_stats": ttest.confidence_interval(confidence_level=0.95)[0],
        "ttest_ci_high_stats": ttest.confidence_interval(confidence_level=0.95)[1],
        "ttest_ci_low": np.abs(standard_ttest_ppf(len(df))),
        "ttest_ci_high": np.abs(standard_ttest_ppf(len(df))),
        "standard_ttest_ppf": standard_ttest_ppf(len(df)),
        "acc": (df[col1] > df[col2]).mean(),
    }
    results.append({
        'Model': "ColBERT (v2)", 
        'col1': "doc1", 
        'col2': "doc2",
        'Paired t-Test Statistic': result['ttest_stats'], 
        'ttest_pvalue': result['ttest_pvalue'],
        'ttest_ci_low_stats': result['ttest_ci_low_stats'],
        'ttest_ci_high_stats': result['ttest_ci_high_stats'],
        'ttest_ci_low': result['ttest_ci_low'],
        'ttest_ci_high': result['ttest_ci_high'],
        'standard_ttest_ppf': result['standard_ttest_ppf'],
        'acc': result['acc'],
        'mean_diff': "NAN",
        'std_diff': "NAN",
        'n': len(df),
        'test': part,
    })

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/250 [00:00<?, ?it/s]


#> QueryTokenizer.tensorize(batch_text[0], batch_background[0], bsize) ==
#> Input: When was House of Angels published?, 		 True, 		 None
#> Output IDs: torch.Size([32]), tensor([ 101,    1, 2043, 2001, 2160, 1997, 7048, 2405, 1029,  102,  103,  103,
         103,  103,  103,  103,  103,  103,  103,  103,  103,  103,  103,  103,
         103,  103,  103,  103,  103,  103,  103,  103], device='cuda:0')
#> Output Mask: torch.Size([32]), tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0], device='cuda:0')



/data2/mohsenfayyaz/projects/Retriever-Contextualization/src/notebooks/Rebuttal/ColBERT/colbert/utils/amp.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast() if self.activated else NullContextManager()


  0%|          | 0/250 [00:00<?, ?it/s]

...

/data2/mohsenfayyaz/projects/Retriever-Contextualization/src/notebooks/Rebuttal/ColBERT/colbert/utils/amp.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast() if self.activated else NullContextManager()


  0%|          | 0/250 [00:00<?, ?it/s]

/data2/mohsenfayyaz/projects/Retriever-Contextualization/src/notebooks/Rebuttal/ColBERT/colbert/utils/amp.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast() if self.activated else NullContextManager()


  0%|          | 0/250 [00:00<?, ?it/s]

/data2/mohsenfayyaz/projects/Retriever-Contextualization/src/notebooks/Rebuttal/ColBERT/colbert/utils/amp.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast() if self.activated else NullContextManager()


In [4]:
print(results)

[{'Model': 'ColBERT (v2)', 'col1': 'doc1', 'col2': 'doc2', 'Paired t-Test Statistic': 13.805361191429894, 'ttest_pvalue': 1.4133858309772357e-32, 'ttest_ci_low_stats': 1.1522051420160206, 'ttest_ci_high_stats': 1.5356698579839794, 'ttest_ci_low': 1.650996151677261, 'ttest_ci_high': 1.650996151677261, 'standard_ttest_ppf': -1.650996151677261, 'acc': 0.82, 'mean_diff': 'NAN', 'std_diff': 'NAN', 'n': 250, 'test': 'brevity_bias'}, {'Model': 'ColBERT (v2)', 'col1': 'doc1', 'col2': 'doc2', 'Paired t-Test Statistic': 15.286235570494666, 'ttest_pvalue': 1.1791567562797535e-37, 'ttest_ci_low_stats': 1.7393994467639933, 'ttest_ci_high_stats': 2.253913053236007, 'ttest_ci_low': 1.650996151677261, 'ttest_ci_high': 1.650996151677261, 'standard_ttest_ppf': -1.650996151677261, 'acc': 0.836, 'mean_diff': 'NAN', 'std_diff': 'NAN', 'n': 250, 'test': 'answer_importance'}, {'Model': 'ColBERT (v2)', 'col1': 'doc1', 'col2': 'doc2', 'Paired t-Test Statistic': 6.337405858864961, 'ttest_pvalue': 1.087749763391